# Assignment 7

Team:

- Bertan Karacora

Tasks:

- Implement the (adapted) TriNet Siamese model:
    - MobileNetV3_small convolutional backbone (until AvgPool) 
    - Fully connected frontend to obtain desired embedding
    - Normalization layer
- Use *Labeled Faces in the Wild* (LFW) dataset (http://vis-www.cs.umass.edu/lfw/)
    - Over 13000 images
    - Train and test set
- Train the following models:
    - TriNet initialized with random weights
    - Fine-tuned TriNet (pretrained MobileNetV3_small and then fine-tuned)
    - The best of the above models, but using semi-hard negative mining strategy (https://arxiv.org/abs/1503.03832)
- Evaluate and compare the model performance (need not be thorough)
- Visualize embeddings (not necessarly for all classes)
- **Extra Point**:
     - Take one or a few images/selfies from the members in your group. Try to use different iluminations, angles, ...
     - Does your model predict similar embeddings for your images?
     - How similar are you with your group partner?
     - To which celebrity does your model think you look like?
    

## Contents

- [x] [Setup](#setup)
    - [x] [Config](#setup_config)
    - [x] [Modules](#setup_modules)
    - [x] [Paths and names](#setup_paths_and_names)
- [x] [Data](#data)
    - [x] [Visualization](#data_visualization)
        - [x] [LFW](#data_visualization_lfw)
    - [x] [Remarks](#data_remarks)
- [x] [Models](#models)
    - [x] [TriNet (adapted)](#models_trinet)
- [x] [Experiments](#experiments)
    - [x] [TriNet on LFW](#experiments_lfw_trinet)
    - [x] [TriNet on LFW (pretrained)](#experiments_lfw_trinet_pretrained)
    - [x] [TriNet on LFW (pretrained) with semi-hard negative mining](#experiments_lfw_trinet_pretrained_shnmining)
    - [x] [Discussion](#experiments_discussion)
- [x] [Bonus](#bonus)

## Setup
<a id="setup"></a>

In [ ]:
%load_ext autoreload
%autoreload 2

### Config
<a id="setup_config"></a>

In [ ]:
import self_supervised_learning_of_depth_and_motion.config as config

config.list_available()

### Modules
<a id="setup_modules"></a>

In [ ]:
from pathlib import Path

import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import torch
import torchinfo
import torchvision as tv

from self_supervised_learning_of_depth_and_motion.evaluation.evaluator import Evaluator
import self_supervised_learning_of_depth_and_motion.libs.factory as factory
import self_supervised_learning_of_depth_and_motion.libs.utils_checkpoints as utils_checkpoints
import self_supervised_learning_of_depth_and_motion.libs.utils_data as utils_data
import self_supervised_learning_of_depth_and_motion.libs.utils_model as utils_model
import self_supervised_learning_of_depth_and_motion.visualization.visualize as visualize

### Paths and names
<a id="setup_paths_and_names"></a>

In [ ]:
name_exp_lfw_unnormalized = "lfw_unnormalized"
name_exp_lfw_trinet = "lfw_trinet"
name_exp_lfw_trinet_pretrained = "lfw_trinet_pretrained"
name_exp_lfw_trinet_pretrained_shnmining = "lfw_trinet_pretrained_shnmining"

path_dir_exp_lfw_trinet = Path(config._PATH_DIR_EXPS) / name_exp_lfw_trinet
path_dir_exp_lfw_trinet_pretrained = Path(config._PATH_DIR_EXPS) / name_exp_lfw_trinet_pretrained
path_dir_exp_lfw_trinet_pretrained_shnmining = Path(config._PATH_DIR_EXPS) / name_exp_lfw_trinet_pretrained_shnmining

## Data
<a id="data"></a>

### Visualization
<a id="data_visualization"></a>

#### LFW
<a id="data_visualization_lfw"></a>

##### Test dataset

![Test dataset](experiments/lfw_trinet/visualizations/Sample_test.png)

##### Validation dataset

![Validate dataset](experiments/lfw_trinet/visualizations/Sample_validation.png)

##### Training dataset

![Training dataset](experiments/lfw_trinet/visualizations/Sample_training.png)

##### Training dataset (normalized)

![Training dataset (normalized)](experiments/lfw_trinet/visualizations/Sample_training_normalized.png)

### Remarks
<a id="data_remarks"></a>

> Implementation:
>
> - [LFW dataset class](assignment/datasets/lfw.py)
> - [Script for computing mean and standard deviation of training dataset](assignment/scripts/compute_mean_and_std.py)

> Remarks:
>
> - Since for most identites in the dataset, there is only a single image available, triplet sampling becomes more difficult. I implemented two approaches, first, simply filtering out all images corresponding to classes that have less than two images (after a stratified train-validation split). Second, using data augmentation. For this, I applied mostly a RandomResizedCrop, RandomHorizontalFlip, RandomRotation, and ColorJitter. The second approach yielded much better progress observable during training.
> - I used deep funneled images as the original website suggested it.
> - The images are normalized using the mean and standard deviation of the training dataset. The normalization is applied also during validation and inference.
> - I forgot to update the mean and std that I have computed when I used the pretrained MobileNet. For this, I should have used the mean and std of the ImageNet dataset.

## Models
<a id="models"></a>

### TriNet (adapted)
<a id="models_trinet"></a>

In [ ]:
config.set_config_exp(path_dir_exp_lfw_trinet)

model = utils_checkpoints.load_model(path_dir_exp_lfw_trinet / "checkpoints" / "best.pth")
print(model)

input_dummy = dict(anchor=torch.zeros(config.MODEL["shape_input"]), positive=torch.zeros(config.MODEL["shape_input"]), negative=torch.zeros(config.MODEL["shape_input"]))
print(torchinfo.summary(model, input_data=[input_dummy], verbose=0, col_names=("input_size", "output_size", "params_percent"), mode="eval"))

### Remarks
<a id="models_remarks"></a>

> Implementation:
>
> - [Training configs](assignment/configs)
> - [TriNet model](assignment/models/siamese.py)

> Remarks:
>
> - I used two linear layers for the fully connected head, as it seemed to perform better during the experiments. However, I did not test this specifically. I would assume it does not make a highly significant difference.
> - For the latent embeddings, I used vectors of length $128$ to encode the original images of shape $3 \cdot 224 \cdot 224$, yielding a compression ration of $0.09%$.

## Experiments
<a id="experiments"></a>

#### Latent space visualizations (flattened test dataset samples as reference)

![Projection_pca_samples](experiments/lfw_trinet/plots/Projection_pca_samples.png)
![Projection_tsne_samples](experiments/lfw_trinet/plots/Projection_tsne_samples.png)

### TriNet on LFW
<a id="experiments_lfw_trinet"></a>

#### Evaluation

In [ ]:
config.set_config_exp(path_dir_exp_lfw_trinet)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset, dataloader = factory.create_dataset_and_dataloader(split="test")

counts_labels = torch.bincount(dataset.targets, minlength=len(dataset.labelset))
topk = torch.topk(counts_labels, 10)

targets_topk = topk.indices
indices_topk = dataset.indices[torch.isin(dataset.targets, targets_topk)]
list_features, list_targets = utils_data.sample_dataset(dataset, indices_topk)

model = utils_checkpoints.load_model(path_dir_exp_lfw_trinet / "checkpoints" / "best.pth")
model = model.to(device)

featuress_anchor = []
latents_anchor = []
targets_anchor = []
with torch.no_grad():
    for features, targets in zip(list_features, list_targets):
        features_anchor = features["anchor"][None, ...]
        target_anchor = torch.as_tensor(targets["anchor"])[None, ...]

        featuress_anchor += [features_anchor]
        targets_anchor += [target_anchor]

        features_anchor = features_anchor.to(device)
        latent_anchor = model.forward_single(features_anchor)
        latents_anchor += [latent_anchor.cpu()]

featuress_anchor_flat = np.concatenate([features_anchor.flatten(1) for features_anchor in featuress_anchor])
featuress_anchor = np.concatenate([utils_data.unnormalize(features_anchor) for features_anchor in featuress_anchor])
targets_anchor = np.concatenate(targets_anchor)
latents_anchor = np.concatenate(latents_anchor)

labels_kmeans_features = KMeans(n_clusters=10).fit_predict(featuress_anchor_flat)
labels_kmeans_latents = KMeans(n_clusters=10).fit_predict(latents_anchor)

ari_imgs = adjusted_rand_score(targets_anchor, labels_kmeans_features)
ari_embs = adjusted_rand_score(targets_anchor, labels_kmeans_latents)

print(f"Clustering images achieves  ARI={round(ari_imgs*100,2)}%")
print(f"Clustering embeddings achieves ARI={round(ari_embs*100,2)}%")

print(f"Compression ratio: {latents_anchor.shape[-1]}/{featuress_anchor_flat.shape[-1]}  = {round(latents_anchor.shape[-1]/featuress_anchor_flat.shape[-1] * 100, 2)}%")

#### Training

![Loss](experiments/lfw_trinet/plots/Loss.png)
![Learning rate](experiments/lfw_trinet/plots/Learning_rate.png)

#### Latent space visualizations

![Projection_pca_latent](experiments/lfw_trinet/plots/Projection_pca_latent.png)
![Projection_tsne_latent](experiments/lfw_trinet/plots/Projection_tsne_latent.png)

### TriNet on LFW (pretrained)
<a id="experiments_lfw_trinet_pretrained"></a>

#### Evaluation

In [ ]:
config.set_config_exp(path_dir_exp_lfw_trinet_pretrained)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset, dataloader = factory.create_dataset_and_dataloader(split="test")

counts_labels = torch.bincount(dataset.targets, minlength=len(dataset.labelset))
topk = torch.topk(counts_labels, 10)

targets_topk = topk.indices
indices_topk = dataset.indices[torch.isin(dataset.targets, targets_topk)]
list_features, list_targets = utils_data.sample_dataset(dataset, indices_topk)

model = utils_checkpoints.load_model(path_dir_exp_lfw_trinet_pretrained / "checkpoints" / "best.pth")
model = model.to(device)

featuress_anchor = []
latents_anchor = []
targets_anchor = []
with torch.no_grad():
    for features, targets in zip(list_features, list_targets):
        features_anchor = features["anchor"][None, ...]
        target_anchor = torch.as_tensor(targets["anchor"])[None, ...]

        featuress_anchor += [features_anchor]
        targets_anchor += [target_anchor]

        features_anchor = features_anchor.to(device)
        latent_anchor = model.forward_single(features_anchor)
        latents_anchor += [latent_anchor.cpu()]

featuress_anchor_flat = np.concatenate([features_anchor.flatten(1) for features_anchor in featuress_anchor])
featuress_anchor = np.concatenate([utils_data.unnormalize(features_anchor) for features_anchor in featuress_anchor])
targets_anchor = np.concatenate(targets_anchor)
latents_anchor = np.concatenate(latents_anchor)

labels_kmeans_features = KMeans(n_clusters=10).fit_predict(featuress_anchor_flat)
labels_kmeans_latents = KMeans(n_clusters=10).fit_predict(latents_anchor)

ari_imgs = adjusted_rand_score(targets_anchor, labels_kmeans_features)
ari_embs = adjusted_rand_score(targets_anchor, labels_kmeans_latents)

print(f"Clustering images achieves  ARI={round(ari_imgs*100,2)}%")
print(f"Clustering embeddings achieves ARI={round(ari_embs*100,2)}%")

print(f"Compression ratio: {latents_anchor.shape[-1]}/{featuress_anchor_flat.shape[-1]}  = {round(latents_anchor.shape[-1]/featuress_anchor_flat.shape[-1] * 100, 2)}%")

#### Training

![Loss](experiments/lfw_trinet_pretrained/plots/Loss.png)
![Learning rate](experiments/lfw_trinet_pretrained/plots/Learning_rate.png)

#### Latent space visualizations

![Projection_pca_latent](experiments/lfw_trinet_pretrained/plots/Projection_pca_latent.png)
![Projection_tsne_latent](experiments/lfw_trinet_pretrained/plots/Projection_tsne_latent.png)

### TriNet on LFW (pretrained) with semi-hard negative mining
<a id="experiments_lfw_trinet_shnmining"></a>

#### Evaluation

In [ ]:
config.set_config_exp(path_dir_exp_lfw_trinet_pretrained_shnmining)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset, dataloader = factory.create_dataset_and_dataloader(split="test")

counts_labels = torch.bincount(dataset.targets, minlength=len(dataset.labelset))
topk = torch.topk(counts_labels, 10)

targets_topk = topk.indices
indices_topk = dataset.indices[torch.isin(dataset.targets, targets_topk)]
list_features, list_targets = utils_data.sample_dataset(dataset, indices_topk)

model = utils_checkpoints.load_model(path_dir_exp_lfw_trinet_pretrained_shnmining / "checkpoints" / "best.pth")
model = model.to(device)

featuress_anchor = []
latents_anchor = []
targets_anchor = []
with torch.no_grad():
    for features, targets in zip(list_features, list_targets):
        features_anchor = features["anchor"][None, ...]
        target_anchor = torch.as_tensor(targets["anchor"])[None, ...]

        featuress_anchor += [features_anchor]
        targets_anchor += [target_anchor]

        features_anchor = features_anchor.to(device)
        latent_anchor = model.forward_single(features_anchor)
        latents_anchor += [latent_anchor.cpu()]

featuress_anchor_flat = np.concatenate([features_anchor.flatten(1) for features_anchor in featuress_anchor])
featuress_anchor = np.concatenate([utils_data.unnormalize(features_anchor) for features_anchor in featuress_anchor])
targets_anchor = np.concatenate(targets_anchor)
latents_anchor = np.concatenate(latents_anchor)

labels_kmeans_features = KMeans(n_clusters=10).fit_predict(featuress_anchor_flat)
labels_kmeans_latents = KMeans(n_clusters=10).fit_predict(latents_anchor)

ari_imgs = adjusted_rand_score(targets_anchor, labels_kmeans_features)
ari_embs = adjusted_rand_score(targets_anchor, labels_kmeans_latents)

print(f"Clustering images achieves  ARI={round(ari_imgs*100,2)}%")
print(f"Clustering embeddings achieves ARI={round(ari_embs*100,2)}%")

print(f"Compression ratio: {latents_anchor.shape[-1]}/{featuress_anchor_flat.shape[-1]}  = {round(latents_anchor.shape[-1]/featuress_anchor_flat.shape[-1] * 100, 2)}%")

#### Training

![Loss](experiments/lfw_trinet_pretrained_shnmining/plots/Loss.png)
![Learning rate](experiments/lfw_trinet_pretrained_shnmining/plots/Learning_rate.png)

#### Latent space visualizations

![Projection_pca_latent](experiments/lfw_trinet_pretrained_shnmining/plots/Projection_pca_latent.png)
![Projection_tsne_latent](experiments/lfw_trinet_pretrained_shnmining/plots/Projection_tsne_latent.png)

### Discussion
<a id="comparison_of_recurrent_models_discussion"></a>

> Implementation:
>
> - [Configs](assignment/configs)
> - [Trainer](assignment/training/trainer_gan.py)
> - [Jupyter notebook for experiments](assignment_7_run.ipynb)
>

> Some remarks:
>
> - The models have been trained for different numbers of epochs. Generally, I had to adapt the learning rate, and training hyperparameters more than one could have expected given the slight changes done in the experiments.
> - A warmup + exponential decay scheduler has been used.
> - It is worth mentioning that the model is very lightweight.
> - No metrics are computed during training. Evaluation is done by computing the Kmeans ARI and quantitatively.
>

> Results:
>

    | Model                                             |   ARI    | 
    | :-----------------------------------------------  | :------: |
    | TriNet                                            |  ~31.85   |
    | TriNet (finetuned)                                |  ~59.21   |
    | TriNet (finetuned) with semi-hard negative mining |  ~76.31   |


> Observations/Conclusion:
>
> - The pretrained MobileNet backbone transfers well to the LFW dataset, as can be seen from the results and the initial loss values. When training with randomly initialized weights, it takes several epochs for the validation loss to start decaying.
> - The semi-hard mining strategy can improve even more upon the finetuned model. Looking at the TSNE and PCA plots with the top 10 most represented individuals, nice clusters can be seen. For less represented classes, this does not look like that. One remaining issue is to maintain a more balanced data sampling. However, for the many less represented individuals, this means more data augmentation and less quality and information in the training data. Therefore, it seems that the LFW dataset is not that perfectly suited for training a Siamese network with the triplet loss.

> 

## Bonus
<a id="bonus"></a>

In [ ]:
path_selfies = Path(config._PATH_DIR_DATA) / "selfies"

selfies = []
for path_selfie in path_selfies.iterdir():
    selfie = tv.io.read_image(path_selfie, apply_exif_orientation=True)
    selfies += [selfie]
selfies = torch.stack(selfies, dim=0)

# Cut top 600 pixels away since to align my face better to the center
selfies = selfies[..., 600:, :]

selfies = tv.transforms.v2.functional.center_crop(selfies, min(*selfies.shape[-2:]))
selfies = tv.transforms.v2.functional.resize(selfies, (256, 256))
selfies = tv.transforms.v2.functional.center_crop(selfies, (224, 224))
selfies = tv.transforms.v2.functional.to_dtype(selfies, dtype=torch.float32, scale=True)

visualize.visualize_images(selfies)

selfies = tv.transforms.v2.functional.normalize(selfies, mean=[0.4614, 0.3953, 0.3480], std=[0.2861, 0.2614, 0.2568])

## Does your model predict similar embeddings for your images?

In [ ]:
config.set_config_exp(path_dir_exp_lfw_trinet_pretrained_shnmining)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = utils_checkpoints.load_model(path_dir_exp_lfw_trinet_pretrained_shnmining / "checkpoints" / "best.pth")
model = model.to(device)

selfies = selfies.to(device)
with torch.no_grad():
    latents = model.forward_single(selfies)

d_selfies_selfies_pairwise = (latents[:, None, ...] - latents[None, ...]).pow(2).sum(dim=2)

print("Distance matrix:")
print(d_selfies_selfies_pairwise)

No, apparently, the images have pairwise distances in latent space that are significantly higher than the targeted margin of 0.2. As we have seen in the previous results, the model performs best for data it has seen a lot, and as we will see below, it leverages the imbalance of the dataset class distribution to lower the loss by focusing on the most common samples only.

## To which celebrity does your model think you look like?

In [ ]:
dataset, dataloader = factory.create_dataset_and_dataloader(split="test")

counts_labels = torch.zeros(len(dataset.labelset), device=device)
sum_d_selfies_anchors = torch.zeros(len(selfies), len(dataset.labelset), device=device)

with torch.no_grad():
    for features, targets in dataloader:
        features_anchor = features["anchor"]
        target_anchor = targets["anchor"]

        features_anchor = features_anchor.to(device)
        target_anchor = target_anchor.to(device)
        latents_anchor = model.forward_single(features_anchor)

        d_selfies_anchor_pairwise = (latents[:, None, ...] - latents_anchor[None, ...]).pow(2).sum(dim=2)

        counts_labels += torch.bincount(target_anchor, minlength=len(dataset.labelset))
        sum_d_selfies_anchors[:, target_anchor] += d_selfies_anchor_pairwise

counts_labels = counts_labels.cpu()
sum_d_selfies_anchors = sum_d_selfies_anchors.cpu()

In [ ]:
mean_d_selfies_anchors = torch.where(counts_labels != 0, sum_d_selfies_anchors / counts_labels, float("inf"))

topk = torch.topk(mean_d_selfies_anchors, 10, largest=False)
values_topk = topk.values
indices_topk = topk.indices

for i, topk_selfie in enumerate(dataset.labelset[indices_topk]):
    print(f"Top 10 for selfie #{i+1}:")
    print(topk_selfie)
    print(values_topk[i])
    print("\n\n")

Conclusion: The imbalance of the available data is probably be the reason for this unexpected behaviour. Two "solutions" come to mind: Removing some samples from the overrepresented classes and using a balanced sampling with data augmentation to mitigate the effect of underrepresented classes. Both have considerable disadvatages, as mentioned before. I guess, entensive tuning would be necessary to optimize this.